# Leaflet cluster map of talk locations

Run this from the _talks/ directory, which contains .md files of all your talks. This scrapes the location YAML field from each .md file, geolocates it with geopy/Nominatim, and uses the getorg library to output data, HTML, and Javascript for a standalone cluster map.

In [22]:
!pip install getorg --upgrade ipywidgets
import glob
import re
import shutil
from pathlib import Path

import getorg
from geopy import Nominatim


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\siser\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [23]:
repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "_talks").exists() and (candidate / "talkmap").exists():
        repo_root = candidate
        break

talk_dir = repo_root / "_talks"
output_dir = repo_root / "talkmap"
other_locations_file = repo_root / "scripts" / "other-locations.txt"

g = sorted(talk_dir.glob("*.md"))

In [ ]:
geocoder = Nominatim(user_agent="siserte-talkmap", timeout=10)
location_dict = {}
location = ""
permalink = ""
title = ""

location_overrides = {
    "Grenoble (France)": "Grenoble, France",
    "PPAM26, Poznan (Poland)": "Poznan, Poland",
    "Poznan (Poland)": "Poznan, Poland",
    "Politechnika Częstochowska, Poland": "Częstochowa, Poland",
    "Technical University of Munich, Garching near Munich, Germany": "Munich, Germany",
    "ISC25, Hamburg, Germany": "Hamburg, Germany",
    "Garching (Germany)": "Garching, Germany",
}


def normalize_location(value):
    if not value:
        return ""
    cleaned = value.strip().replace("’", "'")
    if cleaned in location_overrides:
        return location_overrides[cleaned]
    cleaned = re.sub(r"\s*\(([^)]+)\)\s*$", r", \1", cleaned)
    cleaned = re.sub(r"^[A-Z0-9]+,\s*", "", cleaned)
    cleaned = re.sub(r"\s+near\s+.*$", "", cleaned)
    return cleaned.strip()


def geocode_location(value):
    if not value or value == "Online":
        return None

    normalized = normalize_location(value)
    if not normalized:
        return None

    try:
        result = geocoder.geocode(normalized)
        if result is None:
            print(f"->SKIPPED: no geocode for {value!r} (normalized: {normalized!r})")
        return result
    except Exception as exc:
        print(f"->SKIPPED: geocoding failed for {value!r} (normalized: {normalized!r}): {exc}")
        return None

In [25]:
for file in g:
    with open(file, 'r', encoding="utf-8") as f:
        lines = f.read()
        if 'location: "' in lines:
            loc_start = lines.find('location: "') + 11
            lines_trim = lines[loc_start:]
            loc_end = lines_trim.find('"')
            location = lines_trim[:loc_end]
            if "Teruel" in location:
                print("->IGNORED:", location, "\n")
            elif location and location not in location_dict and location != "Online":
                normalized = normalize_location(location)
                if normalized:
                    geocode = geocode_location(normalized)
                    if geocode is not None:
                        location_dict[normalized] = geocode
                        print(normalized, "\n", geocode)

Madrid, Spain 
 Madrid, Comunidad de Madrid, España
Valladolid, Spain 
 Valladolid, Castilla y León, España
Lugano, Switzerland 
 Lugano, Circolo di Lugano ovest, Distretto di Lugano, Ticino, Schweiz/Suisse/Svizzera/Svizra
Paris, France 
 Paris, Île-de-France, France métropolitaine, France
Barcelona, Spain 
 Barcelona, Barcelonès, Barcelona, Catalunya, España
Castelló de la Plana, Spain 
 Castelló de la Plana / Castellón de la Plana, la Plana Alta, Castelló / Castellón, Comunitat Valenciana, España
Córdoba, Spain 
 Córdoba, Andalucía, España
Timisoara, Romania 
 Timișoara, Timiș, România
Rome, Italy 
 Roma, Roma Capitale, Lazio, Italia
Bristol, United Kingdom 
 City of Bristol, West of England, England, United Kingdom
Málaga, Spain 
 Málaga, Málaga-Costa del Sol, Málaga, Andalucía, España
->IGNORED: Teruel, Spain 

Limassol, Cyprus 
 Λεμεσός, Δήμος Λεμεσού, Επαρχία Λεμεσού, Κύπρος, 3085, Κύπρος - Kıbrıs
Rennes, France 
 Rennes, Ille-et-Vilaine, Bretagne, France métropolitaine, France
P

In [26]:
if other_locations_file.exists():
    with open(other_locations_file, 'r', encoding="utf-8") as f:
        while line := f.readline():
            location = line.rstrip()
            if location and location not in location_dict and location != "Online":
                normalized = normalize_location(location)
                if normalized:
                    geocode = geocode_location(normalized)
                    if geocode is not None:
                        location_dict[normalized] = geocode
                        print(normalized, "\n", geocode)

->SKIPPED: geocoding failed for 'Belfast, Northern Ireland, UK' (normalized: 'Belfast, Northern Ireland, UK'): Non-successful status code 429
->SKIPPED: geocoding failed for 'Berkeley, California, USA' (normalized: 'Berkeley, California, USA'): Non-successful status code 429
->SKIPPED: geocoding failed for 'St. Charles, Illinois, USA' (normalized: 'St. Charles, Illinois, USA'): Non-successful status code 429
->SKIPPED: geocoding failed for 'Fiuggi, Italy' (normalized: 'Fiuggi, Italy'): Non-successful status code 429
->SKIPPED: geocoding failed for 'Amsterdam, Netherlands' (normalized: 'Amsterdam, Netherlands'): Non-successful status code 429
->SKIPPED: geocoding failed for 'Ljubljana, Slovenia' (normalized: 'Ljubljana, Slovenia'): Non-successful status code 429
->SKIPPED: geocoding failed for 'Dallas, Texas, USA' (normalized: 'Dallas, Texas, USA'): Non-successful status code 429
->SKIPPED: geocoding failed for 'Salt Lake City, Utah, USA' (normalized: 'Salt Lake City, Utah, USA'): Non-s

In [27]:
if output_dir.exists():
    shutil.rmtree(output_dir)
else:
    output_dir.mkdir(parents=True, exist_ok=True)

m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name=str(output_dir), hashed_usernames=False)
print(f"Created {len(location_dict)} geocoded locations in {output_dir}")

Created 20 geocoded locations in \\wsl.localhost\Ubuntu\home\siserte\siserte.github.io\talkmap


In [28]:
print(f"Unique geocoded locations: {len(location_dict)}")
print(f"Output folder: {output_dir}")
print("Skipped entries are printed above as ->SKIPPED or ->IGNORED")

Unique geocoded locations: 20
Output folder: \\wsl.localhost\Ubuntu\home\siserte\siserte.github.io\talkmap
Skipped entries are printed above as ->SKIPPED or ->IGNORED
